In [ ]:
import pandas as pd

# Загрузка данных из CSV-файла
csv_file = 'resumes_cleaned.csv'
df = pd.read_csv(csv_file)

print("Столбцы в CSV:", df.columns.tolist())  # Вывод списка столбцов
print(f"Загружено {len(df)} резюме.")  # Вывод количества загруженных резюме

print("\nПервые 5 строк данных:")
print(df.head())

Столбцы в CSV: ['ФИО', 'Желаемая должность', 'Зарплата', 'Личная информация', 'Местоположение', 'Занятость и график', 'Общий опыт', 'Обязанности', 'Ключевые навыки', 'Обо мне', 'Образование', 'Языки', 'Ссылка', 'категория']
Загружено 235 резюме.

Первые 5 строк данных:
        ФИО         Желаемая должность           Зарплата Личная информация  \
0  Кандидат           Senior developer         Не указано   Мужчина, 50 лет   
1  Кандидат           Senior Developer         Не указано  Мужчина, 34 года   
2  Кандидат      .Net Senior developer  300 000 ₽ на руки   Мужчина, 46 лет   
3  Кандидат  Senior Software Developer         Не указано   Мужчина, 59 лет   
4  Кандидат       Senior PHP Developer         Не указано   Мужчина, 31 год   

                                      Местоположение Занятость и график  \
0  Киев , хочу переехать (Москва, Санкт-Петербург...         Не указано   
1  Москва , м. Красные ворота , готов к переезду ...         Не указано   
2  Москва , м. Щукинская , гот

In [ ]:
import re

# Убедимся, что данные загружены
csv_file = 'resumes_cleaned.csv'
df = pd.read_csv(csv_file)

# Приведём текст к нижнему регистру для удобства поиска
df['Ключевые навыки'] = df['Ключевые навыки'].fillna('').str.lower()
df['Обо мне'] = df['Обо мне'].fillna('').str.lower()
df['Обязанности'] = df['Обязанности'].fillna('').str.lower()

# Объединим все текстовые поля, где могут быть навыки
df['text_for_skills'] = (
    df['Ключевые навыки'] + ' ' +
    df['Обо мне'] + ' ' +
    df['Обязанности']
)

# Определим шаблоны для каждого навыка (с учётом синонимов и вариаций)
skill_patterns = {
    'Excel': r'\b(excel|мs excel|майкрософт excel|эксель)\b',
    '1C': r'\b(1[сc]|1-[сc]|один[сc]|1с|1c)\b',  # латинская и кириллическая "с"
    'SQL': r'\b(sql|structured query language|mysql|postgresql|sqlite|mssql|pl/sql)\b',
    'Python': r'\b(python|питон|py)\b',
    'AutoCAD': r'\b(autocad|автокад)\b',
    'SAP': r'\b(sap|sap erp|sap r/3|sap hana)\b',
    'CRM': r'\b(crm|customer relationship management|битрикс24|salesforce|амоcrm|hubspot)\b',
    'Английский B2+': r'\b(английский.*[b-c][1-2]|английский.*продвинут|английский.*upper|английский.*c1|английский.*c2|english.*[b-c][1-2]|english.*advanced|english.*upper|english.*c1|english.*c2)\b'
}

# Создадим столбцы с бинарными метками
for skill, pattern in skill_patterns.items():
    df[skill] = df['text_for_skills'].str.contains(pattern, regex=True, na=False).astype(int)

# Посмотрим, сколько резюме содержат каждый навык
print("Распределение навыков:")
for skill in skill_patterns:
    print(f"{skill}: {df[skill].sum()} из {len(df)}")

Распределение навыков:
Excel: 15 из 235
1C: 12 из 235
SQL: 146 из 235
Python: 53 из 235
AutoCAD: 1 из 235
SAP: 2 из 235
CRM: 17 из 235
Английский B2+: 188 из 235


/tmp/ipython-input-3380635180.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[skill] = df['text_for_skills'].str.contains(pattern, regex=True, na=False).astype(int)
/tmp/ipython-input-3380635180.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[skill] = df['text_for_skills'].str.contains(pattern, regex=True, na=False).astype(int)
/tmp/ipython-input-3380635180.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[skill] = df['text_for_skills'].str.contains(pattern, regex=True, na=False).astype(int)
/tmp/ipython-input-3380635180.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[skill] = df['text_for_skills'].str.cont

In [ ]:
# Создадим копию текста для "очищенной" версии (без утечек)
df['text_clean'] = df['text_for_skills'].copy()

# Маскируем Python и SQL
df['text_clean'] = df['text_clean'].str.replace(r'\b(python|питон|py|sql|mysql|postgresql|sqlite|mssql|pl/sql)\b', '[SKILL]', regex=True, flags=re.IGNORECASE)

# Маскируем должности (пример: любое упоминание "senior", "developer", "инженер" и т.д.)
role_keywords = r'\b(senior|middle|junior|lead|главный|разработчик|developer|engineer|инженер|аналитик|analyst|manager|менеджер|директор|director|специалист|specialist)\b'
df['text_clean'] = df['text_clean'].str.replace(role_keywords, '[ROLE]', regex=True, flags=re.IGNORECASE)

# Проверим, что замена прошла
print("\nПример до маскировки:")
print(df['text_for_skills'].iloc[0][:200])
print("\nПример после маскировки:")
print(df['text_clean'].iloc[0][:200])


Пример до маскировки:
atm, c/c++, device driver, dvb, embedded, kernel, linux, os/2, perl, tcp/ip, video encoding, vxworks, русский — родной, английский — c1 — продвинутый, украинский — c2 — в совершенстве не указано harmo

Пример после маскировки:
atm, c/c++, device driver, dvb, embedded, kernel, linux, os/2, perl, tcp/ip, video encoding, vxworks, русский — родной, английский — c1 — продвинутый, украинский — c2 — в совершенстве не указано harmo


In [ ]:
import nltk
import string
from collections import Counter

# Скачаем необходимые данные для nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [ ]:
def extract_features(text):
    # Общее количество символов
    char_count = len(text)

    # Разбиваем на слова
    words = nltk.word_tokenize(text.lower()) if text.strip() else []
    word_count = len(words)

    # Количество слов с заглавной буквы (в исходном тексте!)
    upper_words = sum(1 for word in text.split() if word.isupper() and word.isalpha())

    # Количество пунктуации
    punct_count = sum(1 for char in text if char in string.punctuation)

    # Средняя длина слова
    avg_word_len = sum(len(word) for word in words) / word_count if word_count > 0 else 0

    # POS-теггинг
    pos_tags = nltk.pos_tag(words)
    pos_counts = Counter(tag for word, tag in pos_tags)

    # Подсчёт частей речи (упрощённо)
    noun_count = sum(1 for word, tag in pos_tags if tag.startswith('NN'))
    verb_count = sum(1 for word, tag in pos_tags if tag.startswith('VB'))
    adj_count = sum(1 for word, tag in pos_tags if tag.startswith('JJ'))
    adv_count = sum(1 for word, tag in pos_tags if tag.startswith('RB'))
    pron_count = sum(1 for word, tag in pos_tags if tag.startswith('PRP'))

    return pd.Series({
        'word_count': word_count,
        'char_count': char_count,
        'avg_word_len': avg_word_len,
        'punct_count': punct_count,
        'upper_words': upper_words,
        'noun_count': noun_count,
        'verb_count': verb_count,
        'adj_count': adj_count,
        'adv_count': adv_count,
        'pron_count': pron_count
    })

# Применяем к исходному тексту (можно и к text_clean, но логичнее к исходному)
df_features = df['text_for_skills'].apply(extract_features)
df = pd.concat([df, df_features], axis=1)

print("\nДополнительные признаки добавлены. Пример:")
print(df[['word_count', 'char_count', 'noun_count', 'verb_count']].head())


Дополнительные признаки добавлены. Пример:
   word_count  char_count  noun_count  verb_count
0        63.0       327.0        42.0         1.0
1       234.0      1342.0       141.0         4.0
2       314.0      1893.0       199.0        12.0
3       192.0      1044.0        71.0        18.0
4        84.0       394.0        50.0         4.0


In [ ]:
target_skill = 'Python'  # можно заменить на любой из списка
print(f"\nАнализируем навык: {target_skill}")
print(f"Баланс классов: {df[target_skill].value_counts().to_dict()}")


Анализируем навык: Python
Баланс классов: {0: 182, 1: 53}


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# Векторизация исходного текста (с утечками)
tfidf_word = TfidfVectorizer(ngram_range=(1, 2), max_features=10000, stop_words='english')
tfidf_char = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=5000)

X_word = tfidf_word.fit_transform(df['text_for_skills'])
X_char = tfidf_char.fit_transform(df['text_for_skills'])

# Объединяем
X_text = hstack([X_word, X_char])

# Векторизация ОЧИЩЕННОГО текста (без утечек)
X_word_clean = tfidf_word.transform(df['text_clean'])  # используем те же векторайзеры!
X_char_clean = tfidf_char.transform(df['text_clean'])
X_text_clean = hstack([X_word_clean, X_char_clean])

# Масштабируем доп признаки
from sklearn.preprocessing import StandardScaler

feature_cols = ['word_count', 'char_count', 'avg_word_len', 'punct_count',
                'upper_words', 'noun_count', 'verb_count', 'adj_count', 'adv_count', 'pron_count']

X_features = df[feature_cols].values
scaler = StandardScaler()
X_features_scaled = scaler.fit_transform(X_features)

print(f"Форма текстовых признаков: {X_text.shape}")
print(f"Форма доп признаков: {X_features_scaled.shape}")

Форма текстовых признаков: (235, 15000)
Форма доп признаков: (235, 10)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix
import numpy as np

y = df[target_skill].values

# Разделим данные
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

X_train_text_clean, X_test_text_clean, _, _ = train_test_split(
    X_text_clean, y, test_size=0.2, random_state=42, stratify=y
)

X_train_feat, X_test_feat, _, _ = train_test_split(
    X_features_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# Объединённые признаки (текст + доп)
X_train_combined = hstack([X_train_text, X_train_feat])
X_test_combined = hstack([X_test_text, X_test_feat])

X_train_combined_clean = hstack([X_train_text_clean, X_train_feat])
X_test_combined_clean = hstack([X_test_text_clean, X_test_feat])

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier

def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else y_pred

    f1 = f1_score(y_test, y_pred, average='macro')
    auc = roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) > 1 else 0.0

    return {
        'model': model_name,
        'F1 (macro)': f1,
        'ROC-AUC': auc
    }

# Список моделей
models = {
    'SVM': SVC(probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Naive Bayes': MultinomialNB(),  # работает только с неотрицательными данными → подходит для TF-IDF
    'MLP': MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
}

# Оценим модели: только текст (с утечками)
results = []
for name, model in models.items():
    try:
        res = evaluate_model(model, X_train_text, X_test_text, y_train, y_test, f"{name} (текст)")
        results.append(res)
    except Exception as e:
        print(f"Ошибка в {name}: {e}")

# Оценим модели: текст + доп признаки
for name, model in models.items():
    try:
        # Naive Bayes не работает с отрицательными значениями → пропустим его для combined
        if name == 'Naive Bayes':
            continue
        res = evaluate_model(model, X_train_combined, X_test_combined, y_train, y_test, f"{name} (текст+фичи)")
        results.append(res)
    except Exception as e:
        print(f"Ошибка в {name} (combined): {e}")

# Оценим модели на ОЧИЩЕННОМ тексте (без утечек)
for name, model in models.items():
    try:
        if name == 'Naive Bayes':
            res = evaluate_model(model, X_train_text_clean, X_test_text_clean, y_train, y_test, f"{name} (очищ. текст)")
        else:
            res = evaluate_model(model, X_train_combined_clean, X_test_combined_clean, y_train, y_test, f"{name} (очищ. + фичи)")
        results.append(res)
    except Exception as e:
        print(f"Ошибка в {name} (clean): {e}")

# Вывод результатов
results_df = pd.DataFrame(results)
print("\nРезультаты моделей:")
print(results_df.round(3).sort_values('F1 (macro)', ascending=False))


Результаты моделей:
                               model  F1 (macro)  ROC-AUC
1              Random Forest (текст)       1.000    1.000
2          Gradient Boosting (текст)       1.000    1.000
7     Gradient Boosting (текст+фичи)       1.000    1.000
6         Random Forest (текст+фичи)       1.000    1.000
11  Gradient Boosting (очищ. + фичи)       0.847    0.965
4                        MLP (текст)       0.774    0.828
13                MLP (очищ. + фичи)       0.774    0.826
8                   MLP (текст+фичи)       0.774    0.866
10      Random Forest (очищ. + фичи)       0.722    0.872
0                        SVM (текст)       0.522    0.992
5                   SVM (текст+фичи)       0.434    0.914
3                Naive Bayes (текст)       0.434    0.869
9                 SVM (очищ. + фичи)       0.434    0.904
12         Naive Bayes (очищ. текст)       0.434    0.747


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from scipy.sparse import hstack

# Общие параметры
MAX_FEATURES_WORD = 10000
MAX_FEATURES_CHAR = 5000

# --- 1. TF-IDF: слова + символы (уже есть, но пересоздадим явно) ---
tfidf_word = TfidfVectorizer(ngram_range=(1, 2), max_features=MAX_FEATURES_WORD, stop_words='english')
tfidf_char = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=MAX_FEATURES_CHAR)

X_word_tfidf = tfidf_word.fit_transform(df['text_for_skills'])
X_char_tfidf = tfidf_char.fit_transform(df['text_for_skills'])
X_text_tfidf_full = hstack([X_word_tfidf, X_char_tfidf])

X_word_clean_tfidf = tfidf_word.transform(df['text_clean'])
X_char_clean_tfidf = tfidf_char.transform(df['text_clean'])
X_text_clean_tfidf_full = hstack([X_word_clean_tfidf, X_char_clean_tfidf])

# --- 2. TF-IDF: только слова ---
tfidf_word_only = TfidfVectorizer(ngram_range=(1, 2), max_features=MAX_FEATURES_WORD, stop_words='english')
X_text_tfidf_word = tfidf_word_only.fit_transform(df['text_for_skills'])
X_text_clean_tfidf_word = tfidf_word_only.transform(df['text_clean'])

# --- 3. CountVectorizer: только слова ---
count_vec = CountVectorizer(ngram_range=(1, 2), max_features=MAX_FEATURES_WORD, stop_words='english')
X_text_count = count_vec.fit_transform(df['text_for_skills'])
X_text_clean_count = count_vec.transform(df['text_clean'])

In [ ]:
def run_experiment(vec_name, X_text, X_text_clean, X_feat, y, feature_cols):
    # Разделение
    X_train_text, X_test_text, y_train, y_test = train_test_split(
        X_text, y, test_size=0.2, random_state=42, stratify=y
    )
    X_train_text_clean, X_test_text_clean, _, _ = train_test_split(
        X_text_clean, y, test_size=0.2, random_state=42, stratify=y
    )
    X_train_feat, X_test_feat, _, _ = train_test_split(
        X_feat, y, test_size=0.2, random_state=42, stratify=y
    )

    # Комбинированные признаки
    X_train_combined = hstack([X_train_text, X_train_feat])
    X_test_combined = hstack([X_test_text, X_test_feat])
    X_train_combined_clean = hstack([X_train_text_clean, X_train_feat])
    X_test_combined_clean = hstack([X_test_text_clean, X_test_feat])

    results = []

    for name, model in models.items():
        # Только текст (с утечками)
        try:
            res = evaluate_model(model, X_train_text, X_test_text, y_train, y_test, f"{name} (текст)")
            res['vectorizer'] = vec_name
            results.append(res)
        except Exception as e:
            print(f"[{vec_name}] Ошибка в {name} (текст): {e}")

        # Текст + фичи (кроме Naive Bayes)
        if name != 'Naive Bayes':
            try:
                res = evaluate_model(model, X_train_combined, X_test_combined, y_train, y_test, f"{name} (текст+фичи)")
                res['vectorizer'] = vec_name
                results.append(res)
            except Exception as e:
                print(f"[{vec_name}] Ошибка в {name} (текст+фичи): {e}")

        # Очищенный текст
        try:
            if name == 'Naive Bayes':
                res = evaluate_model(model, X_train_text_clean, X_test_text_clean, y_train, y_test, f"{name} (очищ. текст)")
            else:
                res = evaluate_model(model, X_train_combined_clean, X_test_combined_clean, y_train, y_test, f"{name} (очищ. + фичи)")
            res['vectorizer'] = vec_name
            results.append(res)
        except Exception as e:
            print(f"[{vec_name}] Ошибка в {name} (clean): {e}")

    return results

In [ ]:
# Целевая переменная
y = df[target_skill].values

# Список моделей (повторим здесь, чтобы функция видела)
models = {
    'SVM': SVC(probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Naive Bayes': MultinomialNB(),
    'MLP': MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
}

# Масштабированные доп признаки (уже есть как X_features_scaled)
X_feat = X_features_scaled

# Запуск
all_results = []

# 1. TF-IDF (слова + символы)
all_results.extend(run_experiment("TF-IDF (words+chars)", X_text_tfidf_full, X_text_clean_tfidf_full, X_feat, y, feature_cols))

# 2. TF-IDF (только слова)
all_results.extend(run_experiment("TF-IDF (words only)", X_text_tfidf_word, X_text_clean_tfidf_word, X_feat, y, feature_cols))

# 3. CountVectorizer (слова)
all_results.extend(run_experiment("CountVectorizer (words)", X_text_count, X_text_clean_count, X_feat, y, feature_cols))

# Вывод
# Собираем результаты в DataFrame
results_df = pd.DataFrame(all_results)

# Упорядочиваем столбцы: сначала vectorizer, потом остальное
results_df = results_df[['vectorizer', 'model', 'F1 (macro)', 'ROC-AUC']]

# Сортируем: сначала по векторизатору, потом по F1 (macro) по убыванию
results_df = results_df.sort_values(['vectorizer', 'F1 (macro)'], ascending=[True, False])

# Выводим ВСЁ в одной строке на запись — без разбивки на блоки
print("\nПолные результаты по всем векторизаторам:")
print(results_df.round(3).to_string(index=False, justify='left'))


Полные результаты по всем векторизаторам:
vectorizer              model                             F1 (macro)  ROC-AUC
CountVectorizer (words)        Gradient Boosting (текст) 1.000       1.000   
CountVectorizer (words)   Gradient Boosting (текст+фичи) 1.000       1.000   
CountVectorizer (words) Gradient Boosting (очищ. + фичи) 0.794       0.926   
CountVectorizer (words)              Naive Bayes (текст) 0.763       0.915   
CountVectorizer (words)            Random Forest (текст) 0.722       1.000   
CountVectorizer (words)        Naive Bayes (очищ. текст) 0.720       0.799   
CountVectorizer (words)       Random Forest (текст+фичи) 0.664       0.989   
CountVectorizer (words)                      MLP (текст) 0.664       0.780   
CountVectorizer (words)                 MLP (текст+фичи) 0.664       0.861   
CountVectorizer (words)               MLP (очищ. + фичи) 0.664       0.818   
CountVectorizer (words)     Random Forest (очищ. + фичи) 0.598       0.852   
CountVectorizer (word